### 1. Causal Language Modeling (CLM) 이란?

#### 정의
> 주어진 이전 토큰들을 바탕으로 **다음 토큰을 예측**하는 작업

#### 작동 원리

```
입력: [토큰1, 토큰2, 토큰3, 토큰4, 토큰5]

1단계 (위치 1): 토큰1 봄 → 토큰2 예측
2단계 (위치 2): 토큰1, 토큰2 봄 → 토큰3 예측
3단계 (위치 3): 토큰1, 토큰2, 토큰3 봄 → 토큰4 예측
4단계 (위치 4): 토큰1, 토큰2, 토큰3, 토큰4 봄 → 토큰5 예측

손실 = 평균(모든 예측 오류)
```

#### 예시: 문장 "hello world today"

```
원문을 토큰화: ["hello", "world", "today"]

훈련 목표:
- [hello] → predict "world"
- [hello, world] → predict "today"

모델이 단순히 다음 단어를 맞추려고 노력하는 과정에서 언어의 구조를 자동으로 배운다
```

### 2. 왜 CLM이 강력한가?

#### 특징 1: 자기감독학습 (Self-supervised Learning)
- 수동으로 레이블링한 데이터 필요 없음
- 인터넷의 모든 텍스트 = 훈련 데이터
- 비용 극도로 낮음

#### 특징 2: 다양한 능력 자동 습득
```
"hello world" 예측 노력
→ 단어 의미 학습
→ 문법 학습
→ 기본 추론 학습
→ (큰 모델) 복잡한 추론 학습
```

#### 특징 3: Fine-tuning 없이도 작동
- Pre-training 후 바로 다양한 태스크 수행
- "모델이 이미 충분히 많이 배웠다"는 의미

### 3. Cross Entropy Loss (교차 엔트로피 손실)

#### 정의
> 모델의 예측 확률분포와 실제 정답 사이의 거리를 측정하는 함수

#### 수식
```
L = -log(P(correct_token))

여기서:
- P(correct_token): 모델이 정답에 할당한 확률
- -log(): 확률이 낮을수록 손실이 커짐
```

#### 직관적 이해

```
시나리오: "hello ___" 다음에 올 단어 예측

모델 예측 (확률):
- "world": 0.7 (70%)
- "there": 0.2 (20%)
- "my": 0.1 (10%)

정답: "world"

손실 = -log(0.7) = 0.36

vs 다른 예측:
- 정답에 0.9 할당 → 손실 = -log(0.9) = 0.11 (더 낮음)
- 정답에 0.1 할당 → 손실 = -log(0.1) = 2.30 (높음)
```

#### 왜 이 손실 함수를 쓰는가?

1. **확률 해석**: 예측값을 확률로 자연스럽게 변환
2. **그래디언트 효과**: 잘못된 예측일수록 크기가 커서 더 강한 업데이트
3. **정보 이론적 근거**: 엔트로피의 최소화 = 불확실성 최소화

### 4. 분산 학습(Distributed Training) 맛보기

#### 수천 개 GPU가 필요한 이유

```
GPT-3 학습:
- 1조 개 토큰 (100,000시간 읽기)
- 단일 GPU: 2년 이상 필요
- 3,072개 GPU: 약 35일 학습
```

#### 분산 학습의 종류

**1) 데이터 병렬 (Data Parallelism)**
```
각 GPU에 모델의 동일한 복사본
데이터만 다르게 분배

GPU1: 배치1 처리
GPU2: 배치2 처리
GPU3: 배치3 처리
→ 결과 평균화 후 가중치 업데이트
```

**2) 텐서 병렬 (Tensor Parallelism)**
```
모델을 여러 GPU에 분산

GPU1: 모델 일부
GPU2: 모델 일부
GPU3: 모델 일부
→ 각 GPU가 부분 계산, 결과 통합
```

**3) 파이프라인 병렬 (Pipeline Parallelism)**
```
모델의 계층을 여러 GPU에 분산

GPU1: 레이어 1-5
GPU2: 레이어 6-10
GPU3: 레이어 11-15
→ 순차적으로 계산 (동시성 낮지만 메모리 효율)
```

### 실습

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

### 문자수준 언어 모델링 코퍼스

In [2]:
# 작은 텍스트 코퍼스
corpus = """hello world hello there
the quick brown fox jumps over the lazy dog
hello there hello world
a quick brown fox
the world is beautiful""".strip()

print(f"코퍼스:\n{corpus}\n")
print(f"전체 문자 수: {len(corpus)}")

# 문자 수준 토큰화
chars = sorted(set(corpus))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"\n고유 문자 수: {len(chars)}")
print(f"문자 목록 (첫 10개): {chars[:10]}")
print(f"\nchar_to_idx 예시: {dict(list(char_to_idx.items())[:10])}")

코퍼스:
hello world hello there
the quick brown fox jumps over the lazy dog
hello there hello world
a quick brown fox
the world is beautiful

전체 문자 수: 132

고유 문자 수: 28
문자 목록 (첫 10개): ['\n', ' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']

char_to_idx 예시: {'\n': 0, ' ': 1, 'a': 2, 'b': 3, 'c': 4, 'd': 5, 'e': 6, 'f': 7, 'g': 8, 'h': 9}


### 훈련데이터 생성

- X 형태: (139, 8) - 139개 배치, 각각 8개 문자 입력
- Y 형태: (139,) - 각 배치의 타겟 문자 (다음 문자)
- block_size=8: Causal Language Model에서 한 번에 처리할 문자 개수

In [3]:
# 토큰화
tokens = [char_to_idx[ch] for ch in corpus]
print(f"토큰화 결과 (처음 50개): {tokens[:50]}")
print(f"\n원문:  {corpus[:50]}")
print(f"토큰:  {tokens[:50]}")

# 훈련 데이터: (입력, 타겟) 쌍
block_size = 8  # 한 번에 8개 문자 처리
X = []  # 입력
Y = []  # 타겟 (다음 문자)

for i in range(len(tokens) - block_size):
    X.append(tokens[i:i+block_size])
    Y.append(tokens[i+block_size])

X = np.array(X)
Y = np.array(Y)

print(f"\n훈련 데이터 크기:")
print(f"  X 형태: {X.shape} (배치, 시퀀스 길이)")
print(f"  Y 형태: {Y.shape} (배치,)")
print(f"\n예시 (처음 3개):")
for i in range(3):
    input_chars = [idx_to_char[idx] for idx in X[i]]
    target_char = idx_to_char[Y[i]]
    print(f"  입력: {''.join(input_chars)} → 타겟: {target_char}")

토큰화 결과 (처음 50개): [9, 6, 13, 13, 16, 1, 24, 16, 19, 13, 5, 1, 9, 6, 13, 13, 16, 1, 21, 9, 6, 19, 6, 0, 21, 9, 6, 1, 18, 22, 10, 4, 12, 1, 3, 19, 16, 24, 15, 1, 7, 16, 25, 1, 11, 22, 14, 17, 20, 1]

원문:  hello world hello there
the quick brown fox jumps 
토큰:  [9, 6, 13, 13, 16, 1, 24, 16, 19, 13, 5, 1, 9, 6, 13, 13, 16, 1, 21, 9, 6, 19, 6, 0, 21, 9, 6, 1, 18, 22, 10, 4, 12, 1, 3, 19, 16, 24, 15, 1, 7, 16, 25, 1, 11, 22, 14, 17, 20, 1]

훈련 데이터 크기:
  X 형태: (124, 8) (배치, 시퀀스 길이)
  Y 형태: (124,) (배치,)

예시 (처음 3개):
  입력: hello wo → 타겟: r
  입력: ello wor → 타겟: l
  입력: llo worl → 타겟: d


### 손실함수 구현(Softmax & Cross Entropy)
**purpose_and_why:** 모델의 예측을 확률로 변환하고 손실을 계산합니다.
- **왜?** Softmax는 로짓→확률 변환, Cross Entropy는 정답 예측 확률로 손실을 측정해 학습 신호를 제공합니다.

**parameter_tensor_explanation:**
- softmax 입력: (배치, 어휘크기) → 출력: (배치, 어휘크기) 확률
- cross_entropy_loss 입력: logits(배치×어휘), targets(배치)
- 출력: 스칼라 손실값 (모든 배치의 평균)

**metaphor_and_blackbox:** 모델이 '여러 후보 중 정답에 얼마나 자신감 있는가'를 측정. 확률이 높을수록 손실은 낮음.


In [4]:
def softmax(logits):
    """소프트맥스 함수: 로짓을 확률로 변환"""
    exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))  # 수치 안정성
    return exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

def cross_entropy_loss(logits, targets):
    """Cross Entropy Loss 계산
    
    logits: (배치, 어휘크기) - 모델의 원점수
    targets: (배치,) - 정답 토큰 인덱스
    """
    # 소프트맥스로 확률 계산
    probs = softmax(logits)
    
    # 배치 크기
    batch_size = logits.shape[0]
    
    # 정답에 할당한 확률
    correct_log_probs = -np.log(probs[np.arange(batch_size), targets] + 1e-8)
    
    # 평균 손실
    loss = np.mean(correct_log_probs)
    
    return loss, probs

# 테스트
logits_example = np.random.randn(4, len(chars)) * 0.01  # 매우 작은 초기값
targets_example = np.array([0, 1, 2, 3])

loss, probs = cross_entropy_loss(logits_example, targets_example)
print("Cross Entropy Loss 테스트")
print(f"  로짓 형태: {logits_example.shape}")
print(f"  타겟: {targets_example}")
print(f"  손실: {loss:.4f}")
print(f"  확률 형태: {probs.shape}")
print(f"  정답 확률 (예시): {probs[0, targets_example[0]]:.4f}")

Cross Entropy Loss 테스트
  로짓 형태: (4, 28)
  타겟: [0 1 2 3]
  손실: 3.3289
  확률 형태: (4, 28)
  정답 확률 (예시): 0.0360


### 모델 파라메터초기화

- W 형태: (어휘크기, 어휘크기) = (23, 23) - 입력 토큰→출력 로짓 변환
- B 형태: (어휘크기,) = (23,) - 편향
- 초기값: 매우 작은 난수로 시작 (학습 안정성을 위해)

In [6]:
chars

['\n',
 ' ',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']